# Exp 28: JPEG quality 10, non-VIP vs all

Card: [exp28_jpeg.md](../../experiments/exp28_jpeg.md)


In [ ]:
import os

import matplotlib.pyplot as plt
import albumentations as A
import numpy as np
from PIL import Image
from sklearn.metrics import average_precision_score, precision_recall_curve

from core.config import CLASSES, ROOT, TRAIN_DIR
from core.cv import VIP, CVResult, Config, CrossValidator
from core.data import read_train
from core.gradcam import GradCamFigures
from core.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-jpeg-smoke",
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    N_SAMPLE = 2  # Cat, Dog images per comparison figure
    N_PREVIEW = 1  # images in JPEG examples figure
    N_POSITIVES = 2  # images per VIP Grad-CAM figure
else:
    OVERRIDES = {}
    N_SAMPLE = 16  # exp19 sample
    N_PREVIEW = 6
    N_POSITIVES = None  # all labeled positives
SEED = 0  # exp19 sample
QUALITIES = [None, 10]  # None: original
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIGS = {
    f"JPEG {rows}": Config.from_toml(
        CONFIG_DIR / "exp28_jpeg" / f"resnet18_{rows.replace('-', '_').lower()}.toml",
        **OVERRIDES,
    )
    for rows in ("non-VIP", "all")
}
EXP20 = Config.from_toml(CONFIG_DIR / "exp20_coarse_dropout_non_vip" / "resnet18.toml")
EXP16 = Config.from_toml(
    CONFIG_DIR / "exp16_downsampling" / "resnet18.toml"
)  # exp19 rows drawn from exp16 OOF
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp28_jpeg"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode
SERIES = {"exp20": "#2a78d6", "JPEG non-VIP": "#1baf7a", "JPEG all": "#eb6834"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)


## JPEG compression examples


In [ ]:
train = read_train()
generic = train[train[VIP].sum(axis=1).eq(0) & train[["Cat", "Dog"]].sum(axis=1).gt(0)]
ids = generic.sample(N_PREVIEW, random_state=SEED).image_id
fig, axes = plt.subplots(
    len(ids),
    len(QUALITIES),
    figsize=(2.6 * len(QUALITIES), 2.8 * len(ids)),
    squeeze=False,
)
for r, image_id in enumerate(ids):
    image = A.Resize(EXP20.img_size, EXP20.img_size)(
        image=np.array(Image.open(TRAIN_DIR / f"{image_id}.jpg").convert("RGB"))
    )["image"]
    for k, q in enumerate(QUALITIES):
        shown = (
            image
            if q is None
            else A.ImageCompression(quality_range=(q, q), p=1)(image=image)["image"]
        )
        axes[r, k].imshow(shown)
        axes[r, k].set_title("original" if q is None else f"quality {q}")
for ax in axes.flat:
    ax.set_axis_off()
fig.tight_layout()
figures = {f"{PREFIX}_examples.png": fig}
plt.show()

## Training


In [ ]:
results = {"exp20": CVResult.load(EXP20)}
results.update({name: CrossValidator(cfg).run() for name, cfg in CONFIGS.items()})

## Grad-CAM Cat, Dog: exp20 vs JPEG non-VIP vs JPEG all


In [ ]:
cams = {"exp20": GradCamFigures(EXP20)} | {
    name: GradCamFigures(cfg) for name, cfg in CONFIGS.items()
}
exp16 = GradCamFigures(EXP16)
for cls in ["Cat", "Dog"]:
    # exp19 rows; smoke OOF holds few rows
    rows = (cams["JPEG all"] if SMOKE else exp16).positive_rows(cls, N_SAMPLE, SEED)
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_compare.png"] = GradCamFigures.compare(cams, cls, rows)
plt.show()

## Grad-CAM Cat, Dog: trained images


In [ ]:
for name in CONFIGS:
    slug = name.removeprefix("JPEG ").replace("-", "_").lower()
    for cls in ["Cat", "Dog"]:
        # exp19 sample: same seed over same out-of-fold rows
        fig = cams[name].positives(cls, train=True, n=N_SAMPLE, seed=SEED)
        figures[f"{PREFIX}_{slug}_grad_cam_{cls.lower()}_train.png"] = fig
plt.show()

## Grad-CAM VIP: unseen vs trained images


In [ ]:
for name in CONFIGS:
    slug = name.removeprefix("JPEG ").replace("-", "_").lower()
    for cls in VIP:
        for train in (False, True):
            fig = cams[name].positives(cls, train=train, n=N_POSITIVES)
            split = "train" if train else "validation"
            figures[f"{PREFIX}_{slug}_grad_cam_{cls.lower()}_{split}.png"] = fig
plt.show()

## Precision-recall curves


In [ ]:
fig, axes = plt.subplots(
    1, len(CLASSES), figsize=(3.4 * len(CLASSES), 3.6), sharey=True
)
for cls, ax in zip(CLASSES, axes):
    for name, r in results.items():
        y, p = r.oof[cls], r.oof[f"p_{cls}"]
        precision, recall, _ = precision_recall_curve(y, p)
        ax.plot(
            recall,
            precision,
            color=SERIES[name],
            lw=2,
            drawstyle="steps-post",
            label=f"{name}, AP {average_precision_score(y, p):.3f}",
        )
    ax.set(
        title=f"{cls} ({int(y.sum())} positives)",
        xlabel="recall",
        xlim=(0, 1.02),
        ylim=(0, 1.05),
    )
    ax.legend(loc="lower left", alignment="left")
axes[0].set_ylabel("precision")
fig.tight_layout()
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

In [ ]:
with Experiment(CONFIGS["JPEG all"].experiment).run("summary", {"val_mode": "fast"}) as run:
    if SAVE_FIGURES:
        run.log_figures(figures, FIGURES)